In [ ]:
!pip install -q openai

In [ ]:
import json
import os
import re
import subprocess
import textwrap
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from statistics import mean, stdev
from typing import Callable, Optional

from openai import OpenAI

OPENAI_API_KEY = ''  # paste your key here
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
client = OpenAI()
MODEL = 'gpt-4o'


# Minimal self-contained agent loop (from agent_harness_core)
def run_agent_loop(
    messages,
    tools,
    tool_handlers,
    system='',
    max_iterations=20,
):
    # messages: list[dict] (num_messages,) -> list[dict] (num_messages + num_turns*2,)
    full = ([{'role': 'system', 'content': system}] + messages) if system else list(messages)
    for _ in range(max_iterations):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=full,
            tools=tools if tools else None,
            tool_choice='auto' if tools else None,
        )
        msg = resp.choices[0].message
        reason = resp.choices[0].finish_reason
        ser = {'role': 'assistant', 'content': msg.content}
        if msg.tool_calls:
            ser['tool_calls'] = [
                {
                    'id': tc.id,
                    'type': 'function',
                    'function': {
                        'name': tc.function.name,
                        'arguments': tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ]
        full.append(ser)
        messages.append(ser)
        if reason != 'tool_calls':
            return messages
        for tc in msg.tool_calls:
            name = tc.function.name
            inp = json.loads(tc.function.arguments)
            h = tool_handlers.get(name)
            try:
                result = h(**inp) if h else f'[ToolError] No handler: {name}'
            except Exception as exc:
                result = f'[ToolError] {type(exc).__name__}: {exc}'
            tm = {'role': 'tool', 'tool_call_id': tc.id, 'content': str(result)}
            full.append(tm)
            messages.append(tm)
    return messages


def lm_call(prompt, system='You are a concise assistant.', max_tokens=500):
    # Minimal single-turn LLM call. Used by all improvement mechanisms.
    # prompt: str -> str (model reply)
    resp = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': prompt},
        ],
    )
    return resp.choices[0].message.content.strip()


print(f'Client ready. Model: {MODEL}')

# 1) Reflexion

## 1.1 Intuition and Motivation

Standard agent loops (Core §1) retry blindly when a task fails: the model
sees the same context and tends to produce the same mistake.

**Reflexion** (Shinn et al., 2023) adds a *verbal gradient*: after each failed
attempt, the agent writes a short natural-language reflection on *what went wrong
and what to do differently*. This reflection is injected into the next attempt
as additional context, steering the model away from the same error.

The key insight: **natural language is a sufficient gradient signal** for
discrete, non-differentiable tasks like tool use and code generation.
No backpropagation. No parameter update. Just a sentence that says
'you tried X, it failed because Y, next time try Z.'

## 1.2 Architecture

```
Attempt 1
  task -> agent_loop -> result
  verifier(result) -> FAIL
  reflect(task, result) -> verbal_reflection_1
         |
         v
Attempt 2
  [verbal_reflection_1] + task -> agent_loop -> result
  verifier(result) -> PASS  -> done
```

The reflection lives in `episodic_memory: list[str] (num_attempts,)`.
Each new attempt injects the full memory so the agent sees all prior failures.

## 1.3 Sample Input / Output

```python
agent = ReflexionAgent(task_verifier=lambda r: 'error' not in r.lower())
result = agent.run(task='Write a Python one-liner that reverses a string.',
                   max_attempts=3)
# Attempt 1: result fails verifier
# Reflection: 'I used [::-1] but forgot to handle None. Next time check input.'
# Attempt 2: passes verifier
# result.attempts = 2, result.passed = True
```

In [ ]:
@dataclass
class AttemptRecord:
    attempt_num: int
    messages: list
    final_reply: str
    passed: bool
    reflection: Optional[str] = None


@dataclass
class ReflexionResult:
    task: str
    # attempts: list[AttemptRecord] (num_attempts,)
    attempts: list
    passed: bool

    @property
    def num_attempts(self):
        return len(self.attempts)

    def summary(self):
        lines = [f'Task: {self.task[:60]}']
        for a in self.attempts:
            status = 'PASS' if a.passed else 'FAIL'
            lines.append(f'  Attempt {a.attempt_num}: {status}')
            if a.reflection:
                lines.append(f'    Reflection: {a.reflection[:80]}')
        lines.append(f'Final: {"PASS" if self.passed else "FAIL"} in {self.num_attempts} attempt(s)')
        return '\n'.join(lines)


class ReflexionAgent:
    # Wraps run_agent_loop with verbal self-reflection on failure.
    # Implements Reflexion (Shinn et al., 2023) without parameter updates.

    _REFLECT_SYSTEM = (
        'You are an expert at diagnosing agent failures. '
        'Given a task and a failed attempt, write one concise sentence explaining '
        'what went wrong and what specific strategy to try differently next time. '
        'Be concrete. Do not repeat the task description.'
    )

    def __init__(
        self,
        task_verifier,
        tools=None,
        tool_handlers=None,
        base_system='You are a helpful assistant.',
    ):
        # task_verifier: Callable(str) -> bool  -- returns True if the attempt passed
        self._verifier = task_verifier
        self._tools = tools or []
        self._handlers = tool_handlers or {}
        self._base_system = base_system
        # episodic_memory: list[str] (num_reflections,) -- grows across attempts
        self.episodic_memory: list[str] = []

    def _build_system(self):
        # Inject all prior reflections into the system prompt.
        # episodic_memory: list[str] (num_reflections,) -> str (augmented system prompt)
        if not self.episodic_memory:
            return self._base_system
        memory_block = '\n'.join(
            f'  - {r}' for r in self.episodic_memory
        )
        return (
            f'{self._base_system}\n\n'
            f'# Prior Attempt Reflections\n'
            f'You have tried this class of task before. Learn from these failures:\n'
            f'{memory_block}'
        )

    def _reflect(self, task, final_reply):
        # Ask the model what went wrong and how to improve.
        # task: str, final_reply: str -> str (verbal reflection, 1 sentence)
        prompt = (
            f'Task: {task}\n\n'
            f'Attempt output (which FAILED verification):\n{final_reply}\n\n'
            f'What went wrong, and what should be tried next time?'
        )
        return lm_call(prompt, system=self._REFLECT_SYSTEM, max_tokens=120)

    def _extract_reply(self, messages):
        # Pull the last assistant text from a messages list.
        # messages: list[dict] (num_messages,) -> str
        return next(
            (m['content'] for m in reversed(messages)
             if m.get('role') == 'assistant' and m.get('content')),
            '(no reply)',
        )

    def run(self, task, max_attempts=3):
        # Run the task with Reflexion: attempt -> verify -> reflect -> retry.
        # task: str, max_attempts: int -> ReflexionResult
        records = []

        for attempt_num in range(1, max_attempts + 1):
            system = self._build_system()
            # Fresh messages per attempt -- context stays clean; reflections via system
            # messages: list[dict] (1,) initially
            messages = [{'role': 'user', 'content': task}]
            run_agent_loop(
                messages=messages,
                tools=self._tools,
                tool_handlers=self._handlers,
                system=system,
            )
            reply = self._extract_reply(messages)
            passed = self._verifier(reply)

            record = AttemptRecord(
                attempt_num=attempt_num,
                messages=messages,
                final_reply=reply,
                passed=passed,
            )

            if passed:
                records.append(record)
                print(f'  [Reflexion] Attempt {attempt_num}: PASS')
                return ReflexionResult(task=task, attempts=records, passed=True)

            # Generate verbal reflection and store in episodic memory
            reflection = self._reflect(task, reply)
            record.reflection = reflection
            self.episodic_memory.append(reflection)
            records.append(record)
            print(f'  [Reflexion] Attempt {attempt_num}: FAIL')
            print(f'    Reflection: {reflection[:80]}')

        return ReflexionResult(task=task, attempts=records, passed=False)


# -- Demo -------------------------------------------------------------------
def _length_verifier(reply):
    # Task passes if the reply is a short Python one-liner (< 80 chars, has 'lambda' or '[:]')
    stripped = reply.strip().split('\n')[-1]
    return len(stripped) < 80 and any(kw in reply for kw in ['lambda', '[::-1]', 'reversed'])

agent_reflexion = ReflexionAgent(
    task_verifier=_length_verifier,
    base_system='You are a Python expert. Answer in one line of code only, no explanation.',
)

result = agent_reflexion.run(
    task='Write a Python one-liner that reverses a string s. Return only the expression.',
    max_attempts=3,
)
print()
print(result.summary())

# 2) Automated Prompt Optimisation

## 2.1 Intuition and Motivation

The system prompt is the most powerful lever you have over agent behaviour,
yet it is typically written once by hand and never updated.
**Automated Prompt Engineering (APE)** and **PromptBreeder** (Fernando et al., 2024)
treat the prompt as a *learnable parameter* and use the model itself to
generate and evaluate improved variants.

The core idea: if you can score a prompt on a task (does it produce correct
answers?), you can do hill-climbing in prompt space using the LLM as both
the mutation operator and the fitness evaluator.

## 2.2 Optimisation Loop

```
Round 0: base_prompt
  |
  v
generate_variants(base_prompt, num_variants)
  -> variants: list[str] (num_variants,)
  |
  v
score_variants(variants, examples)
  -> scores: list[float] (num_variants,)
  |
  v
best_prompt = variants[argmax(scores)]
  |
  v (next round uses best_prompt as base)
Round 1: best_prompt -> ... (repeat num_rounds)
```

## 2.3 Sample Input / Output

```python
opt = PromptOptimiser()
result = opt.optimise(
    base_prompt='Answer the question.',
    examples=[('What is 2+2?', '4'), ('Capital of France?', 'Paris')],
    num_variants=4,
    num_rounds=2,
)
# result.best_prompt: 'Answer concisely and accurately. ...'
# result.score_history: [[0.5, 0.6, 0.4, 0.7], [0.7, 0.8, 0.75, 0.72]]
```

In [ ]:
@dataclass
class OptimisationResult:
    base_prompt: str
    best_prompt: str
    best_score: float
    # score_history: list[list[float]] (num_rounds, num_variants)
    score_history: list
    # prompt_history: list[list[str]] (num_rounds, num_variants)
    prompt_history: list

    def summary(self):
        lines = [
            f'Base prompt score:  {self.score_history[0][0]:.2f}',
            f'Best prompt score:  {self.best_score:.2f}',
            f'Improvement:        +{self.best_score - self.score_history[0][0]:.2f}',
            f'Rounds:             {len(self.score_history)}',
            f'Best prompt (first 100 chars): {self.best_prompt[:100]}',
        ]
        return '\n'.join(lines)


class PromptOptimiser:
    # Implements APE-lite: generate prompt variants, score on examples, select best.
    # Inspired by PromptBreeder (Fernando et al., 2024).

    _MUTATION_SYSTEM = (
        'You are a prompt engineering expert. '
        'Given a system prompt, generate improved variants that will produce '
        'more accurate, concise, and reliable answers. '
        'Vary the wording, structure, and emphasis. '
        'Return exactly the requested number of variants, one per line, prefixed "VARIANT N: ".'
    )

    _JUDGE_SYSTEM = (
        'You are a strict evaluator. Given a task question and the expected answer, '
        'score whether the model answer is correct. '
        'Respond with a single float between 0.0 (wrong) and 1.0 (correct). '
        'No explanation -- just the number.'
    )

    def generate_variants(
        self, base_prompt, num_variants, task_description=''
    ):
        # Generate num_variants improved mutations of base_prompt.
        # base_prompt: str -> variants: list[str] (num_variants,)
        prompt = (
            f'Current system prompt:\n"""{base_prompt}"""\n\n'
            f'Task context: {task_description}\n\n'
            f'Generate {num_variants} improved variants.'
        )
        raw = lm_call(prompt, system=self._MUTATION_SYSTEM, max_tokens=600)
        variants = []
        for line in raw.splitlines():
            m = re.match(r'VARIANT\s*\d+:\s*(.*)', line.strip())
            if m:
                variants.append(m.group(1).strip())
        # Pad with base_prompt if parsing produced fewer than requested
        while len(variants) < num_variants:
            variants.append(base_prompt)
        return variants[:num_variants]

    def score_variant(self, prompt, examples):
        # Evaluate one prompt on all examples; return mean correctness score.
        # prompt: str
        # examples: list[tuple[str, str]] (num_examples,) -- (question, expected_answer)
        # -> float (mean score across num_examples)
        scores = []
        for question, expected in examples:
            answer = lm_call(question, system=prompt, max_tokens=100)
            judge_prompt = (
                f'Question: {question}\n'
                f'Expected: {expected}\n'
                f'Model answer: {answer}\n'
                f'Score (0.0-1.0):'
            )
            raw_score = lm_call(judge_prompt, system=self._JUDGE_SYSTEM, max_tokens=10)
            try:
                score = float(re.findall(r'\d+\.?\d*', raw_score)[0])
                scores.append(min(1.0, max(0.0, score)))
            except (IndexError, ValueError):
                scores.append(0.0)
        # scores: list[float] (num_examples,) -> float (mean)
        return mean(scores) if scores else 0.0

    def optimise(
        self,
        base_prompt,
        examples,
        num_variants=4,
        num_rounds=2,
        task_description='',
    ):
        # Main optimisation loop: generate -> score -> select -> repeat.
        # base_prompt: str
        # examples: list[tuple[str, str]] (num_examples,)
        # num_variants: int, num_rounds: int
        # -> OptimisationResult
        current_best_prompt = base_prompt
        current_best_score = self.score_variant(base_prompt, examples)
        score_history = [[current_best_score]]
        prompt_history = [[base_prompt]]

        for round_num in range(num_rounds):
            print(f'  [PromptOpt] Round {round_num + 1}/{num_rounds} (current best: {current_best_score:.2f})')
            variants = self.generate_variants(
                current_best_prompt, num_variants, task_description
            )
            # variants: list[str] (num_variants,) -> scores: list[float] (num_variants,)
            scores = [self.score_variant(v, examples) for v in variants]
            best_idx = scores.index(max(scores))
            round_best_score = scores[best_idx]
            print(f'  [PromptOpt] Variant scores: {[round(s, 2) for s in scores]}')
            if round_best_score > current_best_score:
                current_best_score = round_best_score
                current_best_prompt = variants[best_idx]
                print(f'  [PromptOpt] New best: {current_best_score:.2f}')
            score_history.append(scores)
            prompt_history.append(variants)

        return OptimisationResult(
            base_prompt=base_prompt,
            best_prompt=current_best_prompt,
            best_score=current_best_score,
            score_history=score_history,
            prompt_history=prompt_history,
        )


# -- Demo -------------------------------------------------------------------
examples = [
    ('What is the capital of France?', 'Paris'),
    ('What is 12 * 8?', '96'),
    ('Name one primary colour.', 'red'),
    ('What does HTTP stand for?', 'HyperText Transfer Protocol'),
]

optimiser = PromptOptimiser()
opt_result = optimiser.optimise(
    base_prompt='Answer the question.',
    examples=examples,
    num_variants=3,
    num_rounds=2,
    task_description='Short factual question answering',
)
print()
print(opt_result.summary())

# 3) Skill Verification and Pruning

## 3.1 Intuition and Motivation

In Core §5, skills are loaded on demand from SKILL.md files. In Core §5.3,
new skills are auto-generated from experience. But skills **rot**: APIs change,
library versions update, the codebase evolves. A skill written three months ago
may now give incorrect instructions.

**Skill verification** proactively tests each skill by:
1. Extracting runnable test cases from the skill's markdown
2. Executing them in a subprocess sandbox
3. Checking that outputs match expectations

**Skill pruning** removes or quarantines skills that fail verification,
preventing the agent from loading stale knowledge.

This closes the quality loop of the skill library:
*create (§5.3) -> verify -> prune -> create again*.

## 3.2 Skill Test Format

Skills include a `## Tests` section with runnable assertions:
```markdown
## Tests
```python
assert subprocess.run('git status', shell=True).returncode == 0
assert 'pytest' in subprocess.run('pytest --version', ...).stdout
```
```

## 3.3 Sample Input / Output

```python
verifier = SkillVerifier()
report = verifier.audit_library(skill_lib)
# VerificationReport:
#   pytest.md    -> PASS (1/1 tests)
#   git.md       -> PASS (2/2 tests)
#   docker.md    -> FAIL (0/1 tests) -- docker not installed
#   stale_api.md -> NO_TESTS
verifier.prune_stale(skill_lib, report, fail_threshold=1)
# Removed: docker.md (quarantined to skills/quarantine/)
```

In [ ]:
from enum import Enum, auto
import shutil


class VerificationStatus(Enum):
    PASS     = auto()
    FAIL     = auto()
    NO_TESTS = auto()
    ERROR    = auto()


@dataclass
class SkillVerificationResult:
    skill_name: str
    status: VerificationStatus
    tests_run: int = 0
    tests_passed: int = 0
    error_message: Optional[str] = None

    @property
    def pass_rate(self):
        return self.tests_passed / self.tests_run if self.tests_run > 0 else 0.0

    def __str__(self):
        if self.status == VerificationStatus.NO_TESTS:
            return f'{self.skill_name:20}: NO_TESTS'
        if self.status == VerificationStatus.ERROR:
            return f'{self.skill_name:20}: ERROR -- {self.error_message}'
        passed = f'{self.tests_passed}/{self.tests_run}'
        return f'{self.skill_name:20}: {self.status.name} ({passed} tests)'


@dataclass
class VerificationReport:
    # results: list[SkillVerificationResult] (num_skills,)
    results: list
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

    def summary(self):
        lines = ['Verification Report:']
        lines += [f'  {r}' for r in self.results]
        passed = sum(1 for r in self.results if r.status == VerificationStatus.PASS)
        failed = sum(1 for r in self.results if r.status == VerificationStatus.FAIL)
        lines.append(f'  Total: {passed} pass, {failed} fail, {len(self.results)} audited')
        return '\n'.join(lines)


class SkillVerifier:
    # Extracts and runs test cases embedded in SKILL.md files.
    # Modelled on Voyager skill self-verification (Wang et al., 2023).

    _TEST_BLOCK_RE = re.compile(
        r'##\s+Tests?\s*\n```python\n(.*?)```', re.DOTALL | re.IGNORECASE
    )
    TIMEOUT_S = 10

    def _extract_tests(self, skill_content):
        # Parse the ## Tests code block from skill markdown.
        # skill_content: str -> list[str] (num_test_lines,)
        m = self._TEST_BLOCK_RE.search(skill_content)
        if not m:
            return []
        code = m.group(1).strip()
        return [line.strip() for line in code.splitlines() if line.strip()]

    def _run_test_line(self, line):
        # Execute one assert-style test line in a subprocess.
        # line: str -> bool (True if test passed)
        script = f'import subprocess\n{line}'
        try:
            r = subprocess.run(
                ['python3', '-c', script],
                capture_output=True, text=True, timeout=self.TIMEOUT_S,
            )
            return r.returncode == 0
        except subprocess.TimeoutExpired:
            return False
        except Exception:
            return False

    def verify_skill(self, skill_name, skill_content):
        # Verify one skill: extract tests, run each, record pass/fail.
        # skill_name: str, skill_content: str -> SkillVerificationResult
        test_lines = self._extract_tests(skill_content)
        if not test_lines:
            return SkillVerificationResult(
                skill_name=skill_name, status=VerificationStatus.NO_TESTS
            )
        # test_lines: list[str] (num_tests,) -> results: list[bool] (num_tests,)
        results = [self._run_test_line(line) for line in test_lines]
        num_passed = sum(results)
        status = VerificationStatus.PASS if all(results) else VerificationStatus.FAIL
        return SkillVerificationResult(
            skill_name=skill_name,
            status=status,
            tests_run=len(results),
            tests_passed=num_passed,
        )

    def audit_library(self, skills_dir):
        # Verify every skill in a directory.
        # skills_dir: str | Path -> VerificationReport
        path = Path(skills_dir)
        results = []
        for skill_file in sorted(path.glob('*.md')):
            content = skill_file.read_text(encoding='utf-8')
            result = self.verify_skill(skill_file.stem, content)
            results.append(result)
        return VerificationReport(results=results)

    def prune_stale(
        self, skills_dir, report, fail_threshold=1
    ):
        # Move failed skills to a quarantine subdirectory.
        # fail_threshold: int -- quarantine if tests_passed < fail_threshold
        # -> list[str] (names of quarantined skills)
        path = Path(skills_dir)
        quarantine = path / 'quarantine'
        quarantine.mkdir(exist_ok=True)
        pruned = []
        for r in report.results:
            if r.status == VerificationStatus.FAIL and r.tests_passed < fail_threshold:
                src = path / f'{r.skill_name}.md'
                dst = quarantine / f'{r.skill_name}.md'
                if src.exists():
                    shutil.move(str(src), str(dst))
                    pruned.append(r.skill_name)
                    print(f'  [SkillVerifier] Quarantined: {r.skill_name}')
        return pruned


# -- Demo: seed skills with and without tests -------------------------------
SKILLS_DIR = Path('skills_test')
SKILLS_DIR.mkdir(exist_ok=True)

(SKILLS_DIR / 'python_version.md').write_text(
    '# Python Version Skill\n'
    '## When to use\nCheck Python version.\n'
    '## Steps\n1. Run python3 --version\n'
    '## Tests\n'
    '```python\n'
    'import subprocess\n'
    'r = subprocess.run(["python3", "--version"], capture_output=True, text=True)\n'
    'assert r.returncode == 0\n'
    'assert "Python" in r.stdout + r.stderr\n'
    '```\n',
    encoding='utf-8',
)

(SKILLS_DIR / 'fake_api.md').write_text(
    '# Fake API Skill\n'
    '## When to use\nCall the FakeAPI service.\n'
    '## Steps\n1. curl https://fakeapi.example.com\n'
    '## Tests\n'
    '```python\n'
    'import subprocess\n'
    'r = subprocess.run(["curl", "https://fakeapi.example.com"], '
    'capture_output=True, timeout=5)\n'
    'assert r.returncode == 0  # will fail -- host does not exist\n'
    '```\n',
    encoding='utf-8',
)

(SKILLS_DIR / 'no_tests.md').write_text(
    '# General Tips\n## When to use\nAlways.\n## Steps\n1. Be careful.\n',
    encoding='utf-8',
)

verifier = SkillVerifier()
report = verifier.audit_library(SKILLS_DIR)
print(report.summary())
pruned = verifier.prune_stale(SKILLS_DIR, report, fail_threshold=1)
print(f'\nPruned: {pruned}')

# 4) Meta-Agent and ADAS

## 4.1 Intuition and Motivation

So far, everything that improves is *inside* the agent: its reflections,
its prompts, its skills. **ADAS** (Automated Design of Agentic Systems,
Hu et al., 2025) asks a deeper question: what if we optimise the *design
of the agent itself* -- its system prompt, tool set, and loop structure?

A **meta-agent** (the optimiser) generates candidate child agent configurations,
evaluates each on a benchmark, and keeps the best. This is evolutionary
search in configuration space, where the LLM is the mutation operator.

The key difference from prompt optimisation (§2):
- §2 optimises one string (the system prompt)
- §4 optimises a full configuration: system prompt + tool selection + iteration budget

## 4.2 Evolutionary Loop

```
Generation 0: base_config
   |
   v
meta_agent generates num_configs variants of base_config
   -> configs: list[AgentConfig] (num_configs,)
   |
   v
evaluate each config on benchmark_tasks
   -> scores: list[float] (num_configs,)
   |
   v
elite = top-k configs by score
   |
   v (next generation uses elite as parents)
Generation 1: mutate(elite) -> ... (repeat num_generations)
```

## 4.3 Sample Input / Output

```python
meta = MetaAgentOptimiser(benchmark_tasks=['What is 2+2?', 'Reverse hello'])
result = meta.run(base_config, num_configs=4, num_generations=2)
# Generation 1: scores [0.5, 0.75, 0.6, 0.4]  best=0.75
# Generation 2: scores [0.8, 0.75, 0.7, 0.85]  best=0.85
# result.best_config.system_prompt: 'You are a precise...'
```

In [ ]:
@dataclass
class AgentConfig:
    # A complete agent configuration that can be evaluated and mutated.
    system_prompt: str
    max_iterations: int = 5
    description: str = ''

    def to_dict(self):
        return {
            'system_prompt': self.system_prompt,
            'max_iterations': self.max_iterations,
            'description': self.description,
        }


@dataclass
class MetaOptResult:
    base_config: AgentConfig
    best_config: AgentConfig
    best_score: float
    # generation_scores: list[list[float]] (num_generations, num_configs)
    generation_scores: list

    def summary(self):
        lines = [f'Meta-Agent Optimisation Result:']
        for g, scores in enumerate(self.generation_scores):
            lines.append(
                f'  Gen {g + 1}: scores={[round(s, 2) for s in scores]} '
                f'best={max(scores):.2f}'
            )
        lines.append(f'Overall best score: {self.best_score:.2f}')
        lines.append(f'Best config (first 80 chars): {self.best_config.system_prompt[:80]}')
        return '\n'.join(lines)


class MetaAgentOptimiser:
    # Evolutionary search over AgentConfig space.
    # Meta-agent generates variants; each variant is evaluated on benchmark tasks.
    # Inspired by ADAS (Hu et al., 2025).

    _META_SYSTEM = (
        'You are an agent architect. Given an agent configuration, generate '
        'improved variants with different system prompts and iteration budgets. '
        'Each variant should take a different strategic approach. '
        'Return exactly the requested number of JSON objects, one per line, '
        'with keys: system_prompt (str), max_iterations (int, 3-10), description (str).'
    )

    def __init__(self, benchmark_tasks, benchmark_verifier=None):
        # benchmark_tasks: list[str] (num_examples,)
        # benchmark_verifier: Callable(str) -> float | None (uses LLM judge if None)
        self._tasks = benchmark_tasks
        self._verifier = benchmark_verifier

    def _lm_judge(self, task, reply):
        # LLM-as-judge: score a reply on a task (0.0 - 1.0).
        # task: str, reply: str -> float
        raw = lm_call(
            f'Task: {task}\nReply: {reply}\n'
            f'Score the reply quality 0.0 (bad) to 1.0 (excellent). Return only the number.',
            system='You are a strict quality judge. Return only a float.',
            max_tokens=10,
        )
        try:
            return float(re.findall(r'\d+\.?\d*', raw)[0])
        except (IndexError, ValueError):
            return 0.0

    def evaluate_config(self, config):
        # Run a config on all benchmark tasks and return mean score.
        # config: AgentConfig -> float (mean score across num_examples)
        scores = []
        for task in self._tasks:
            messages = [{'role': 'user', 'content': task}]
            run_agent_loop(
                messages=messages,
                tools=[],
                tool_handlers={},
                system=config.system_prompt,
                max_iterations=config.max_iterations,
            )
            reply = next(
                (m['content'] for m in reversed(messages)
                 if m.get('role') == 'assistant' and m.get('content')),
                '',
            )
            score = (
                self._verifier(reply)
                if self._verifier
                else self._lm_judge(task, reply)
            )
            scores.append(score)
        # scores: list[float] (num_examples,) -> float (mean)
        return mean(scores) if scores else 0.0

    def generate_variants(self, parent_config, num_configs):
        # Ask the meta-agent to generate num_configs mutations of parent_config.
        # parent_config: AgentConfig -> variants: list[AgentConfig] (num_configs,)
        prompt = (
            f'Parent config:\n{json.dumps(parent_config.to_dict(), indent=2)}\n\n'
            f'Generate {num_configs} improved variants as JSON objects, one per line.'
        )
        raw = lm_call(prompt, system=self._META_SYSTEM, max_tokens=800)
        configs = []
        for line in raw.splitlines():
            line = line.strip()
            if not line.startswith('{'):
                continue
            try:
                d = json.loads(line)
                configs.append(AgentConfig(**{k: d[k] for k in AgentConfig.__dataclass_fields__ if k in d}))
            except (json.JSONDecodeError, TypeError):
                pass
        while len(configs) < num_configs:
            configs.append(parent_config)
        return configs[:num_configs]

    def run(self, base_config, num_configs=4, num_generations=2):
        # Evolutionary optimisation: generate -> evaluate -> select -> repeat.
        # base_config: AgentConfig -> MetaOptResult
        current_best = base_config
        current_best_score = self.evaluate_config(base_config)
        print(f'  [Meta] Base score: {current_best_score:.2f}')
        all_gen_scores = []

        for gen in range(num_generations):
            variants = self.generate_variants(current_best, num_configs)
            # variants: list[AgentConfig] (num_configs,) -> scores: list[float] (num_configs,)
            scores = [self.evaluate_config(v) for v in variants]
            all_gen_scores.append(scores)
            best_idx = scores.index(max(scores))
            print(f'  [Meta] Gen {gen + 1}: {[round(s, 2) for s in scores]} best={max(scores):.2f}')
            if scores[best_idx] > current_best_score:
                current_best_score = scores[best_idx]
                current_best = variants[best_idx]

        return MetaOptResult(
            base_config=base_config,
            best_config=current_best,
            best_score=current_best_score,
            generation_scores=all_gen_scores,
        )


# -- Demo -------------------------------------------------------------------
benchmark = [
    'What is the square root of 144?',
    'Name the largest planet in our solar system.',
    'What programming language is known for its use in data science?',
]

base = AgentConfig(
    system_prompt='Answer the question.',
    max_iterations=3,
    description='baseline',
)

meta = MetaAgentOptimiser(benchmark_tasks=benchmark)
meta_result = meta.run(base, num_configs=3, num_generations=2)
print()
print(meta_result.summary())

# 5) Automated Research Loop

## 5.1 Intuition and Motivation

The most powerful form of self-improvement is **automated scientific inquiry**:
the agent proposes hypotheses, designs experiments to test them, runs the
experiments, interprets results, and refines its understanding.
This is the research cycle formalised as an agent loop.

Unlike Reflexion (which reacts to a single failure) or prompt optimisation
(which searches a fixed objective), the research loop is *open-ended*:
the agent decides what to investigate next based on what it has learned.

## 5.2 The Research Cycle

```
topic (str)
   |
   v
propose_hypothesis(topic, prior_observations)
   -> Hypothesis(statement, prediction, experiment_code)
   |
   v
run_experiment(experiment_code)
   -> ExperimentResult(stdout, exit_code, duration_ms)
   |
   v
analyse_results(hypothesis, result)
   -> AnalysisRecord(confirmed, insight, next_question)
   |
   v
(next iteration uses insight as new observation)
```

## 5.3 Sample Input / Output

```python
loop = ResearchLoop()
report = loop.run(topic='Python list comprehension vs map() performance', max_iterations=3)
# Iteration 1: hypothesis='list comp is faster for simple transforms'
#   experiment: timeit list comp vs map
#   result: list comp: 0.8ms, map: 1.1ms
#   analysis: confirmed, insight: 'list comp ~27% faster for this case'
# Iteration 2: hypothesis='gap narrows for complex transforms'
#   ....
```

In [ ]:
@dataclass
class Hypothesis:
    statement: str       # what the agent believes
    prediction: str      # what the experiment should show if true
    experiment_code: str # runnable Python to test the hypothesis


@dataclass
class ExperimentResult:
    stdout: str
    stderr: str
    exit_code: int
    duration_ms: float

    @property
    def succeeded(self):
        return self.exit_code == 0

    @property
    def output(self):
        return (self.stdout + self.stderr).strip()[:2_000]


@dataclass
class AnalysisRecord:
    hypothesis: Hypothesis
    result: ExperimentResult
    confirmed: bool
    insight: str        # what was learned
    next_question: str  # what to investigate next


@dataclass
class ResearchReport:
    topic: str
    # records: list[AnalysisRecord] (num_iterations,)
    records: list

    def summary(self):
        lines = [f'Research Report: {self.topic}']
        for i, r in enumerate(self.records):
            status = 'CONFIRMED' if r.confirmed else 'REFUTED'
            lines.append(f'  Iter {i+1} [{status}]: {r.hypothesis.statement[:60]}')
            lines.append(f'    Insight: {r.insight[:80]}')
        return '\n'.join(lines)


class ResearchLoop:
    # Autonomous research cycle: hypothesise -> experiment -> analyse -> repeat.
    # The agent designs and runs its own experiments using subprocess.

    _HYPOTHESIS_SYSTEM = (
        'You are a rigorous researcher. Given a topic and prior observations, '
        'propose a specific, falsifiable hypothesis. '
        'Then write a self-contained Python script (no external deps beyond stdlib) '
        'that tests it. The script must print numeric or clear textual results. '
        'Respond in JSON with keys: statement, prediction, experiment_code.'
    )

    _ANALYSIS_SYSTEM = (
        'You are a scientist analysing experiment results. '
        'Given a hypothesis and its experiment output, determine if the hypothesis '
        'was confirmed or refuted. Extract the key numerical insight. '
        'Respond in JSON with keys: confirmed (bool), insight (str), next_question (str).'
    )

    TIMEOUT_S = 15

    def propose_hypothesis(self, topic, observations):
        # Generate a testable hypothesis and experiment code for the topic.
        # topic: str, observations: list[str] (num_prior_insights,) -> Hypothesis
        obs_text = '\n'.join(f'- {o}' for o in observations) if observations else 'None yet.'
        prompt = (
            f'Research topic: {topic}\n\n'
            f'Prior observations:\n{obs_text}\n\n'
            f'Propose the next hypothesis to test.'
        )
        raw = lm_call(prompt, system=self._HYPOTHESIS_SYSTEM, max_tokens=600)
        # Parse JSON from model output
        try:
            clean = re.sub(r'^```json?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
            d = json.loads(clean)
            return Hypothesis(
                statement=d.get('statement', ''),
                prediction=d.get('prediction', ''),
                experiment_code=d.get('experiment_code', 'print("no code")'),
            )
        except (json.JSONDecodeError, KeyError):
            return Hypothesis(statement=raw[:100], prediction='', experiment_code='print("parse error")')

    def run_experiment(self, hypothesis):
        # Execute the experiment code in a subprocess sandbox.
        # hypothesis.experiment_code: str -> ExperimentResult
        start = time.time()
        try:
            r = subprocess.run(
                ['python3', '-c', hypothesis.experiment_code],
                capture_output=True, text=True, timeout=self.TIMEOUT_S,
            )
            duration = (time.time() - start) * 1000
            return ExperimentResult(
                stdout=r.stdout, stderr=r.stderr,
                exit_code=r.returncode, duration_ms=round(duration, 1),
            )
        except subprocess.TimeoutExpired:
            return ExperimentResult(
                stdout='', stderr='Timeout', exit_code=-1,
                duration_ms=self.TIMEOUT_S * 1000,
            )

    def analyse_results(self, hypothesis, result):
        # Interpret experiment output against the hypothesis.
        # -> AnalysisRecord (confirmed, insight, next_question)
        prompt = (
            f'Hypothesis: {hypothesis.statement}\n'
            f'Prediction: {hypothesis.prediction}\n'
            f'Experiment output:\n{result.output}\n'
            f'Exit code: {result.exit_code}\n\n'
            f'Analyse the result.'
        )
        raw = lm_call(prompt, system=self._ANALYSIS_SYSTEM, max_tokens=400)
        try:
            clean = re.sub(r'^```json?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
            d = json.loads(clean)
            return AnalysisRecord(
                hypothesis=hypothesis,
                result=result,
                confirmed=bool(d.get('confirmed', False)),
                insight=d.get('insight', ''),
                next_question=d.get('next_question', ''),
            )
        except (json.JSONDecodeError, KeyError):
            return AnalysisRecord(
                hypothesis=hypothesis, result=result,
                confirmed=False, insight=raw[:100], next_question='',
            )

    def run(self, topic, max_iterations=3):
        # Full autonomous research loop.
        # topic: str, max_iterations: int -> ResearchReport
        observations = []
        records = []
        for iteration in range(max_iterations):
            print(f'  [Research] Iteration {iteration + 1}/{max_iterations}')
            hypothesis = self.propose_hypothesis(topic, observations)
            print(f'    Hypothesis: {hypothesis.statement[:70]}')
            result = self.run_experiment(hypothesis)
            print(f'    Experiment: exit={result.exit_code} {result.duration_ms:.0f}ms')
            print(f'    Output: {result.output[:60]}')
            analysis = self.analyse_results(hypothesis, result)
            print(f'    Confirmed: {analysis.confirmed} | Insight: {analysis.insight[:60]}')
            records.append(analysis)
            observations.append(analysis.insight)

        return ResearchReport(topic=topic, records=records)


# -- Demo -------------------------------------------------------------------
research_loop = ResearchLoop()
report = research_loop.run(
    topic='Python list comprehension vs for-loop performance for simple transforms',
    max_iterations=2,
)
print()
print(report.summary())

# 6) Darwin Godel Machine

*Research note -- no implementation code. This section explains the concept
and points to the paper. Implementing it safely requires containerisation
that is out of scope for a notebook.*

## 6.1 Core Idea

The Darwin Godel Machine (DGM, Zhang et al., 2025) is the most radical
form of self-improvement studied so far: **the agent modifies its own
source code**, not just its prompts or skills.

Starting from a single coding agent, the DGM:
1. Reads its own codebase
2. Proposes modifications (new tools, better loop logic, improved prompts)
3. Evaluates each modification on a coding benchmark (SWE-bench, Polyglot)
4. Keeps modifications that improve benchmark score
5. Uses the improved agent as the base for the next generation

**Result:** automatically improved from 20.0% to 50.0% on SWE-bench
without any human-written code changes.

## 6.2 Why It Is Different from Earlier Sections

| Mechanism | What changes | Persists? |
|-----------|-------------|-----------|
| Reflexion (§1) | Episodic memory | Session only |
| Prompt optimisation (§2) | System prompt string | Yes |
| Skill library (§3) | SKILL.md files | Yes |
| Meta-agent (§4) | Agent configuration | Yes |
| Research loop (§5) | Knowledge (insights) | Yes |
| **DGM (§6)** | **Agent source code** | **Yes -- and recursively** |

The DGM is the only mechanism where the improvement operator itself improves.
Each generation's agent is better at *finding improvements* than the prior one.

## 6.3 Why Not Implement It Here

Safe DGM requires:
- Docker or Singularity isolation (agent runs in a container it cannot escape)
- A ground-truth benchmark with an automated oracle (SWE-bench tests)
- Rollback: if a modification crashes the agent, restore from checkpoint
- Multi-hour compute per generation

The conceptual building blocks are all in your trilogy:
- Source code editing = bash_tool + file_write_tool (Core §2)
- Benchmarking = SkillVerifier pattern (§3 above) applied to the agent itself
- Generation tracking = WorktreeManager (Core §12) -- one worktree per variant
- Evaluation = MetaAgentOptimiser.evaluate_config (§4 above)

## 6.4 Further Reading

- Zhang et al., *Darwin Godel Machine: Open-Ended Evolution of Self-Improving Agents*, arXiv 2505.22954, 2025
- Fernando et al., *PromptBreeder: Self-Referential Self-Improvement via Prompt Evolution*, ICML 2024
- Yin et al., *Godel Agent*, 2024 -- self-referential architecture inspired by Godel Machines
- DARWIN (2026): containerised self-rewriting with beam search -- arXiv 2602.05848

# 7) Self-Rewarding Evaluation

## 7.1 Intuition and Motivation

All quality improvement so far relies on an *external* signal: a task verifier,
a human-labelled example, or a benchmark oracle.
**Self-rewarding LMs** (Yuan et al., 2025) remove this dependency:
**the model evaluates its own outputs** and uses that signal to select
or refine responses.

This is possible because frontier models have strong metacognitive ability
-- they can often tell when a response is good or bad, even if they cannot
always produce a good response on the first try.

The key design challenge: **self-scoring bias**. Models tend to score their
own outputs too generously. Mitigation strategies used here:
- Structured rubric (forces the model to justify the score dimension by dimension)
- Best-of-N selection (generate N candidates, pick the highest-scored)
- Iterative refinement (generate, score, refine, score again)

## 7.2 Sample Input / Output

```python
evaluator = SelfRewardingEvaluator()
result = evaluator.generate_and_select(
    task='Explain gradient descent in two sentences for a high school student.',
    n_candidates=4,
)
# Candidates scored: [3.2, 4.1, 2.8, 4.5]
# Selected: 'Gradient descent is like...' (score=4.5)
```

In [ ]:
@dataclass
class ScoredCandidate:
    text: str
    score: float   # 1.0 - 5.0
    rubric_scores: dict  # dimension -> score


@dataclass
class SelfRewardResult:
    task: str
    # candidates: list[ScoredCandidate] (num_candidates,)
    candidates: list
    selected: ScoredCandidate
    num_refinement_rounds: int = 0

    def summary(self):
        scores = [round(c.score, 2) for c in self.candidates]
        return (
            f'Candidates: {len(self.candidates)} | '
            f'Scores: {scores} | '
            f'Best: {self.selected.score:.1f} | '
            f'Selected (first 80): {self.selected.text[:80]}'
        )


class SelfRewardingEvaluator:
    # Generates multiple response candidates and uses the model itself to score them.
    # Implements Best-of-N and iterative refinement from Yuan et al., 2025.

    # Rubric: structured dimensions to reduce self-scoring bias.
    # Forcing the model to score each dimension separately improves calibration.
    _SCORE_SYSTEM = (
        'You are a strict, calibrated quality evaluator. '
        'Score a response on four dimensions (1-5 each):\n'
        '  accuracy: is the response factually correct?\n'
        '  clarity:  is it easy to understand?\n'
        '  conciseness: is it appropriately brief?\n'
        '  helpfulness: does it address the task fully?\n'
        'Respond in JSON: {"accuracy": N, "clarity": N, "conciseness": N, "helpfulness": N, "reasoning": "..."}.'
        ' Be honest and critical -- do not inflate scores.'
    )

    _REFINE_SYSTEM = (
        'You are an expert writer. Improve the given response based on the quality feedback. '
        'Preserve what is good; fix what is weak. Be concise.'
    )

    def generate_candidates(self, task, n_candidates, system='You are a helpful assistant.'):
        # Sample n_candidates responses for the same task at temperature > 0.
        # task: str -> candidates: list[str] (num_candidates,)
        candidates = []
        for _ in range(n_candidates):
            resp = client.chat.completions.create(
                model=MODEL,
                max_tokens=300,
                temperature=0.8,  # diversity in candidates
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': task},
                ],
            )
            candidates.append(resp.choices[0].message.content.strip())
        return candidates

    def score_candidate(self, task, candidate_text):
        # Self-score one candidate using the structured rubric.
        # task: str, candidate_text: str -> ScoredCandidate
        prompt = f'Task: {task}\n\nResponse to evaluate:\n{candidate_text}'
        raw = lm_call(prompt, system=self._SCORE_SYSTEM, max_tokens=200)
        try:
            clean = re.sub(r'^```json?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
            d = json.loads(clean)
            dims = {k: float(d[k]) for k in ('accuracy', 'clarity', 'conciseness', 'helpfulness') if k in d}
            # rubric_scores: dict (4 dimensions) -> float (mean)
            overall = mean(dims.values()) if dims else 3.0
            return ScoredCandidate(
                text=candidate_text, score=round(overall, 2), rubric_scores=dims
            )
        except (json.JSONDecodeError, ValueError):
            return ScoredCandidate(text=candidate_text, score=3.0, rubric_scores={})

    def generate_and_select(self, task, n_candidates=4, base_system='You are a helpful assistant.'):
        # Best-of-N: generate n_candidates, score all, return highest-scored.
        # task: str, n_candidates: int -> SelfRewardResult
        raw_candidates = self.generate_candidates(task, n_candidates, system=base_system)
        # raw_candidates: list[str] (num_candidates,) -> scored: list[ScoredCandidate] (num_candidates,)
        scored = [self.score_candidate(task, c) for c in raw_candidates]
        best = max(scored, key=lambda c: c.score)
        return SelfRewardResult(task=task, candidates=scored, selected=best)

    def iterative_refinement(self, task, max_rounds=2, base_system='You are a helpful assistant.'):
        # Generate -> score -> refine -> score again, up to max_rounds.
        # Each round the model sees its own score and the rubric and improves.
        # task: str, max_rounds: int -> SelfRewardResult
        current = lm_call(task, system=base_system, max_tokens=300)
        current_scored = self.score_candidate(task, current)
        all_candidates = [current_scored]

        for round_num in range(max_rounds):
            rubric_text = json.dumps(current_scored.rubric_scores, indent=2)
            refine_prompt = (
                f'Task: {task}\n\n'
                f'Current response:\n{current_scored.text}\n\n'
                f'Quality scores (1-5):\n{rubric_text}\n\n'
                f'Improve this response to score higher on the weak dimensions.'
            )
            refined = lm_call(refine_prompt, system=self._REFINE_SYSTEM, max_tokens=300)
            refined_scored = self.score_candidate(task, refined)
            all_candidates.append(refined_scored)
            print(f'  [SelfReward] Round {round_num + 1}: {current_scored.score:.1f} -> {refined_scored.score:.1f}')
            if refined_scored.score > current_scored.score:
                current_scored = refined_scored

        best = max(all_candidates, key=lambda c: c.score)
        return SelfRewardResult(
            task=task, candidates=all_candidates, selected=best, num_refinement_rounds=max_rounds
        )


# -- Demo -------------------------------------------------------------------
evaluator = SelfRewardingEvaluator()

print('Best-of-N selection (4 candidates):')
bon_result = evaluator.generate_and_select(
    task='Explain what a neural network is in exactly two sentences for a 16-year-old.',
    n_candidates=4,
)
print(f'  {bon_result.summary()}')

print()
print('Iterative refinement (2 rounds):')
refine_result = evaluator.iterative_refinement(
    task='Explain what gradient descent does in one paragraph.',
    max_rounds=2,
)
print(f'  Final score: {refine_result.selected.score:.1f}')
print(f'  Response: {refine_result.selected.text[:150]}')

# 8) Capstone: LearningAgent

## 8.1 Architecture

`LearningAgent` wires all seven mechanisms into a single entry point.
The design principle: **each mechanism fires only when relevant**.
Not every task needs all seven -- unnecessary calls waste tokens and time.

```
learn_and_run(task)
   |
   |-- (1) verify skills before starting        [§3 SkillVerifier]
   |
   |-- (2) attempt with Reflexion               [§1 ReflexionAgent]
   |         |                                                    
   |         v PASS on attempt 1: skip to (4)                    
   |         v FAIL: reflect, retry up to max_attempts            
   |
   |-- (3) if novel + complex: propose research [§5 ResearchLoop]
   |         |-- 1 iteration only (lightweight probe)            
   |         |-- insight stored in agent memory                   
   |
   |-- (4) self-reward: best-of-2 selection     [§7 SelfRewardingEvaluator]
   |
   |-- (5) periodically: optimise prompt        [§2 PromptOptimiser]
   |         |-- every N tasks (configurable)                     
   |         |-- uses accumulated task/answer pairs as examples   
   |
   |-- (6) periodically: generate skill from task [Core §5.3]
   |
   v
   final reply (best self-scored candidate)
```

## 8.2 What Persists Across Sessions

| Artefact | Location | Grows from |
|----------|----------|-----------|
| `episodic_memory` | in-memory (session) | Reflexion reflections |
| `skills/` directory | disk | Successful complex tasks |
| `best_system_prompt` | in-memory (session) | Prompt optimisation |
| `research_insights` | in-memory (session) | Research loop |
| `task_examples` | in-memory (session) | Every completed task |

In [ ]:
from pathlib import Path


class LearningAgent:
    # Capstone: composes all self-improvement mechanisms from §1-§7.
    # Each mechanism is gated by a condition so it fires only when useful.

    _SKILL_CREATOR_SYSTEM = (
        'You extract reusable skills from agent task transcripts. '
        'Write a concise SKILL.md with sections: '
        '# Skill Name, ## When to use, ## Steps (numbered). '
        'Under 150 words. Return only the markdown, no preamble.'
    )

    def __init__(
        self,
        base_system='You are a helpful expert assistant.',
        skills_dir='skills_learning',
        prompt_opt_every_n=5,
        min_tool_calls_for_skill=2,
    ):
        # base_system: str -- starting system prompt (improved by §2 over time)
        self.current_system = base_system
        self.skills_dir = Path(skills_dir)
        self.skills_dir.mkdir(exist_ok=True)
        self._prompt_opt_every_n = prompt_opt_every_n
        self._min_tool_calls_for_skill = min_tool_calls_for_skill

        # Persistent state (within session)
        # episodic_memory: list[str] (num_reflections,)
        self.episodic_memory: list = []
        # task_examples: list[tuple[str,str]] (num_completed_tasks,)
        self.task_examples: list = []
        # research_insights: list[str] (num_insights,)
        self.research_insights: list = []
        self._task_count = 0

        # Sub-components
        self._verifier = SkillVerifier()
        self._evaluator = SelfRewardingEvaluator()
        self._opt = PromptOptimiser()
        self._research = ResearchLoop()
        self._reflexion = ReflexionAgent(
            task_verifier=self._default_verifier,
            base_system=base_system,
        )

    def _default_verifier(self, reply):
        # Default pass criterion: non-empty reply with >= 10 words.
        return len(reply.split()) >= 10

    def _audit_skills(self):
        # Verify all skills in skills_dir; prune stale ones.
        report = self._verifier.audit_library(self.skills_dir)
        self._verifier.prune_stale(self.skills_dir, report, fail_threshold=1)
        return report

    def _try_generate_skill(self, task, final_reply, tool_call_count):
        # Auto-generate a SKILL.md if the task was complex enough.
        # Fires when tool_call_count >= min_tool_calls_for_skill.
        if tool_call_count < self._min_tool_calls_for_skill:
            return
        content = lm_call(
            f'Task: {task}\nOutcome: {final_reply[:300]}',
            system=self._SKILL_CREATOR_SYSTEM,
            max_tokens=200,
        )
        first_line = content.strip().splitlines()[0]
        name = 'auto_' + re.sub(r'[^a-z0-9]', '_', first_line.replace('#','').strip().lower())[:25]
        (self.skills_dir / f'{name}.md').write_text(content, encoding='utf-8')
        print(f'  [LearningAgent] Generated skill: {name}.md')

    def _try_optimise_prompt(self):
        # Optimise system prompt every N tasks if we have enough examples.
        if self._task_count % self._prompt_opt_every_n != 0:
            return
        if len(self.task_examples) < 4:
            return
        print(f'  [LearningAgent] Running prompt optimisation ({len(self.task_examples)} examples)...')
        examples = self.task_examples[-8:]  # use most recent 8
        result = self._opt.optimise(
            base_prompt=self.current_system,
            examples=examples,
            num_variants=3,
            num_rounds=1,
        )
        if result.best_score > self._opt.score_variant(self.current_system, examples):
            self.current_system = result.best_prompt
            self._reflexion._base_system = self.current_system
            print(f'  [LearningAgent] Prompt updated. Score: {result.best_score:.2f}')

    def _try_research(self, task, reply):
        # If the task seems novel and the reply mentions uncertainty,
        # run one research iteration to deepen understanding.
        uncertainty_signals = ['not sure', 'unclear', 'it depends', 'however', 'but']
        if not any(s in reply.lower() for s in uncertainty_signals):
            return
        topic = f'Investigating: {task[:60]}'
        report = self._research.run(topic=topic, max_iterations=1)
        for record in report.records:
            self.research_insights.append(record.insight)
        print(f'  [LearningAgent] Research insight: {record.insight[:80]}')

    def learn_and_run(
        self,
        task,
        max_reflexion_attempts=2,
        n_self_reward_candidates=2,
        run_skill_audit=False,
    ):
        # Main entry point. Run all improvement mechanisms as appropriate.
        # task: str -> str (best self-scored reply)
        self._task_count += 1
        print(f'\n[LearningAgent] Task {self._task_count}: {task[:60]}')

        # Optionally audit skills before starting
        if run_skill_audit:
            self._audit_skills()

        # Reflexion: attempt with verbal memory of prior failures
        self._reflexion.episodic_memory = self.episodic_memory
        reflex_result = self._reflexion.run(task, max_attempts=max_reflexion_attempts)
        self.episodic_memory = self._reflexion.episodic_memory
        base_reply = reflex_result.attempts[-1].final_reply

        # Self-reward: generate an extra candidate and pick the better one
        if n_self_reward_candidates > 1:
            print(f'  [LearningAgent] Self-reward selection ({n_self_reward_candidates} candidates)...')
            sr_result = self._evaluator.generate_and_select(
                task=task, n_candidates=n_self_reward_candidates,
                base_system=self.current_system,
            )
            # Compare with Reflexion reply by self-scoring it too
            reflex_scored = self._evaluator.score_candidate(task, base_reply)
            final_reply = (
                sr_result.selected.text
                if sr_result.selected.score > reflex_scored.score
                else base_reply
            )
        else:
            final_reply = base_reply

        # Store as example for future prompt optimisation
        self.task_examples.append((task, final_reply[:200]))

        # Auto-generate skill if task was complex
        tool_calls = sum(
            1 for m in reflex_result.attempts[-1].messages
            if m.get('role') == 'tool'
        )
        self._try_generate_skill(task, final_reply, tool_calls)

        # Probe uncertain topics with a research loop
        self._try_research(task, final_reply)

        # Periodically optimise system prompt
        self._try_optimise_prompt()

        return final_reply

    def status(self):
        return (
            f'LearningAgent Status:\n'
            f'  Tasks completed:       {self._task_count}\n'
            f'  Episodic memories:     {len(self.episodic_memory)}\n'
            f'  Research insights:     {len(self.research_insights)}\n'
            f'  Task examples stored:  {len(self.task_examples)}\n'
            f'  Skills in library:     {len(list(self.skills_dir.glob("*.md")))}\n'
            f'  Current prompt (first 80): {self.current_system[:80]}'
        )


# -- Capstone Demo ----------------------------------------------------------
print('Initialising LearningAgent...')
agent_learning = LearningAgent(
    base_system='You are a helpful, precise assistant. Be concise and accurate.',
    prompt_opt_every_n=10,   # disable auto-optimisation for demo speed
)

tasks = [
    'What is the time complexity of binary search? Explain briefly.',
    'In one paragraph, explain what a transformer attention head computes.',
    'Name three real-world use cases for graph neural networks.',
]

for task in tasks:
    reply = agent_learning.learn_and_run(
        task,
        max_reflexion_attempts=2,
        n_self_reward_candidates=2,
    )
    print(f'  Final reply: {reply[:100]}\n')

print(agent_learning.status())

# 9) User Modeling

## 9.1 Intuition and Motivation

All mechanisms in §1-§8 make the agent better at **tasks** in general.
Hermes's *'grows with you'* claim is a different axis entirely:
the agent builds a persistent model of **you specifically** --
your communication style, your domain, your preferences, your habits.

Three mechanisms work together:

```
Every turn                                     Periodic
────────────────────────────────────────────   ────────────────────────
MemoryNudge                                    DialecticLoop
  After each reply, ask:                         Challenge the current
  'Is there anything about this user            profile for contradictions
   worth remembering?'                           and refine it.
  If yes -> extract -> UserProfile                                    
                                               
UserProfile
  Structured store: style, domain,
  preferences, raw facts
  Injected into system prompt every turn
```

**Why this is different from skill creation (§5.3):**
Skill creation learns reusable *task procedures*.
User modeling learns *who this human is*.
A skill says 'here is how to run pytest'.
A user model says 'this user is a PhD student, prefers bullet points,
works in NLP, and gets annoyed when you add preamble'.

**Honcho dialectic loop (Plastic Labs, 2024):**
The dialectic challenges the model's own beliefs about the user.
Example: the profile says 'user prefers terse answers' but turn 7 shows
they asked for more detail. The dialectic surfaces this contradiction and
produces a refined belief: 'user prefers terse answers *for factual queries*
but wants elaboration for *design decisions*'.

## 9.2 Sample Input / Output

```python
modeler = UserModeler()

modeler.observe(turn={
    'user': 'just give me the command, no explanation',
    'assistant': 'pip install torch'
})
modeler.observe(turn={
    'user': 'what does __init__ do exactly? i keep forgetting',
    'assistant': '__init__ is the constructor...'
})

print(modeler.profile.summary())
# style:       direct, dislikes preamble
# domain:      Python / ML
# preferences: terse for commands, detailed for concepts

# Injected into system prompt:
print(modeler.to_system_block())
# '# User Profile\n- Prefers terse command answers...'
```

In [ ]:
@dataclass
class UserProfile:
    # Structured model of a specific user built from observed interactions.
    # Persisted to disk as USER_PROFILE.json between sessions.

    profile_path: str = '.user_profile.json'

    # Preference dimensions -- each is a list of observed signals
    # style_signals: list[str] (num_observations,)
    style_signals:       list = field(default_factory=list)
    # domain_signals: list[str] (num_observations,)
    domain_signals:      list = field(default_factory=list)
    # preference_signals: list[str] (num_observations,)
    preference_signals:  list = field(default_factory=list)
    # raw_facts: list[str] (num_facts,) -- verbatim remembered items
    raw_facts:           list = field(default_factory=list)
    # contradiction_log: list[str] (num_contradictions,)
    contradiction_log:   list = field(default_factory=list)

    # Refined beliefs: updated by DialecticLoop
    # beliefs: dict[str, str] (num_beliefs,) -- dimension -> refined belief
    beliefs: dict = field(default_factory=dict)

    def add_fact(self, fact):
        # fact: str -> None  (deduplicates by substring match)
        if not any(fact.lower() in f.lower() for f in self.raw_facts):
            self.raw_facts.append(fact)
            self._save()

    def add_signal(self, dimension, signal):
        # dimension: 'style' | 'domain' | 'preference', signal: str -> None
        target = getattr(self, f'{dimension}_signals', None)
        if target is not None and signal not in target:
            target.append(signal)
            self._save()

    def _save(self):
        Path(self.profile_path).write_text(
            json.dumps(self.__dict__, indent=2), encoding='utf-8'
        )

    @classmethod
    def load(cls, path='.user_profile.json'):
        # Load from disk if it exists; otherwise return empty profile.
        p = Path(path)
        if not p.exists():
            return cls(profile_path=path)
        d = json.loads(p.read_text())
        return cls(**d)

    def to_system_block(self):
        # Serialise profile as a system prompt section.
        # -> str (injected into system prompt; empty if no signals yet)
        parts = []
        if self.beliefs:
            parts.append('Refined beliefs about this user:')
            for dim, belief in self.beliefs.items():
                parts.append(f'  {dim}: {belief}')
        elif self.raw_facts:
            parts.append('What I know about this user:')
            # raw_facts: list[str] (num_facts,) -> str (bulleted)
            parts += [f'  - {f}' for f in self.raw_facts[-8:]]
        if not parts:
            return ''
        return '# User Profile\n' + '\n'.join(parts)

    def summary(self):
        return (
            f'UserProfile:\n'
            f'  Style signals:      {len(self.style_signals)}\n'
            f'  Domain signals:     {len(self.domain_signals)}\n'
            f'  Preference signals: {len(self.preference_signals)}\n'
            f'  Raw facts:          {len(self.raw_facts)}\n'
            f'  Refined beliefs:    {len(self.beliefs)}\n'
            f'  Contradictions:     {len(self.contradiction_log)}'
        )


class MemoryNudge:
    # After each conversation turn, asks the LLM whether anything is worth
    # storing about the user. If yes, extracts structured signals and stores them.
    # Direct implementation of Hermes memory nudge pattern.

    _NUDGE_SYSTEM = (
        'You observe human-AI conversations and extract signals about the human. '
        'Given one conversation turn, decide if there is anything worth remembering. '
        'Respond in JSON with keys:\n'
        '  worth_remembering: bool,\n'
        '  facts: list[str]  (specific facts: name, role, project, timezone),\n'
        '  style: list[str]  (communication style signals: terse, formal, impatient),\n'
        '  domain: list[str] (subject areas: ML, fintech, Python),\n'
        '  preferences: list[str] (explicit preferences: no preamble, bullet points).\n'
        'Be conservative -- only flag genuinely reusable signals. '
        'Return {"worth_remembering": false} if nothing stands out.'
    )

    def __init__(self, profile):
        # profile: UserProfile -- mutated in-place on each nudge
        self._profile = profile
        # _nudge_count: int -- total nudges run (for logging)
        self._nudge_count = 0

    def nudge(self, user_message, assistant_reply):
        # Inspect one turn and update the profile if signals are found.
        # user_message: str, assistant_reply: str -> bool (True if anything stored)
        self._nudge_count += 1
        turn_text = f'User: {user_message}\nAssistant: {assistant_reply}'
        raw = lm_call(turn_text, system=self._NUDGE_SYSTEM, max_tokens=250)
        try:
            clean = re.sub(r'^```json?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
            d = json.loads(clean)
        except (json.JSONDecodeError, AttributeError):
            return False

        if not d.get('worth_remembering', False):
            return False

        # Extract and store each signal type
        for fact in d.get('facts', []):
            self._profile.add_fact(fact)
        for signal in d.get('style', []):
            self._profile.add_signal('style', signal)
        for signal in d.get('domain', []):
            self._profile.add_signal('domain', signal)
        for signal in d.get('preferences', []):
            self._profile.add_signal('preference', signal)

        return True


# -- Demo: observe two turns and inspect what was stored -------------------
profile = UserProfile(profile_path='.demo_user_profile.json')
nudge   = MemoryNudge(profile)

turns = [
    ('just give me the bash command, skip the explanation',
     'git log --oneline -10'),
    ('i work on NLP systems at a startup, we use pytorch',
     'Got it. I can tailor examples to PyTorch and NLP contexts.'),
    ('can you always reply in bullet points? easier to skim',
     'Sure, I will use bullet points from now on.'),
]

print('Running memory nudges...')
for user_msg, asst_reply in turns:
    stored = nudge.nudge(user_msg, asst_reply)
    print(f'  stored={stored}: "{user_msg[:50]}"')

print()
print(profile.summary())
print()
print('System block preview:')
print(profile.to_system_block()[:300])

In [ ]:
class DialecticLoop:
    # Periodically challenges the current user model for internal contradictions
    # and produces refined, more nuanced beliefs.
    # Based on the Honcho dialectic pattern (Plastic Labs, 2024).

    # Minimum number of signals before dialectic is worth running
    MIN_SIGNALS = 4

    _CHALLENGE_SYSTEM = (
        'You are a rigorous belief auditor. '
        'Given a set of observations about a user, identify contradictions or '
        'oversimplifications. For each one, propose a refined, more nuanced belief. '
        'Respond in JSON: {"contradictions": [{"raw": str, "refined": str, "dimension": str}]}. '
        'Only flag genuine contradictions, not just different contexts. '
        'If no contradictions exist, return {"contradictions": []}.'
    )

    _SYNTHESIS_SYSTEM = (
        'You are a user modeling expert. '
        'Given all observations and refined beliefs about a user, '
        'write a concise structured belief for each dimension: '
        'style, domain, preferences, context. '
        'Each belief should be one sentence, specific and actionable. '
        'Respond in JSON: {"style": str, "domain": str, "preferences": str, "context": str}. '
        'If insufficient data for a dimension, omit it.'
    )

    def __init__(self, profile):
        # profile: UserProfile -- beliefs dict updated in-place
        self._profile = profile

    def _all_signals(self):
        # Collect all raw signals into a single readable block for the LLM.
        # -> str (formatted signal list)
        p = self._profile
        # signals: dict[str, list[str]] (4 dimensions) -> str
        lines = []
        if p.raw_facts:
            lines.append('Facts: ' + '; '.join(p.raw_facts))
        if p.style_signals:
            lines.append('Style: ' + '; '.join(p.style_signals))
        if p.domain_signals:
            lines.append('Domain: ' + '; '.join(p.domain_signals))
        if p.preference_signals:
            lines.append('Preferences: ' + '; '.join(p.preference_signals))
        if p.contradiction_log:
            lines.append('Prior contradictions: ' + '; '.join(p.contradiction_log[-3:]))
        return '\n'.join(lines)

    def _total_signals(self):
        p = self._profile
        return (
            len(p.style_signals) + len(p.domain_signals)
            + len(p.preference_signals) + len(p.raw_facts)
        )

    def should_run(self):
        # Only run if we have enough observations to make it worthwhile.
        # -> bool
        return self._total_signals() >= self.MIN_SIGNALS

    def challenge(self):
        # Step 1: find contradictions in the current signal set.
        # -> list[dict] (num_contradictions,) each with raw, refined, dimension
        if not self.should_run():
            return []
        all_signals = self._all_signals()
        raw = lm_call(all_signals, system=self._CHALLENGE_SYSTEM, max_tokens=400)
        try:
            clean = re.sub(r'^```json?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
            d = json.loads(clean)
            contradictions = d.get('contradictions', [])
            for c in contradictions:
                self._profile.contradiction_log.append(c.get('raw', ''))
            return contradictions
        except (json.JSONDecodeError, KeyError):
            return []

    def synthesise(self):
        # Step 2: produce refined, actionable beliefs from all observations.
        # Updates profile.beliefs in-place.
        # -> dict[str, str] (num_beliefs,) dimension -> refined belief
        if not self.should_run():
            return {}
        all_signals = self._all_signals()
        raw = lm_call(all_signals, system=self._SYNTHESIS_SYSTEM, max_tokens=300)
        try:
            clean = re.sub(r'^```json?|```$', '', raw.strip(), flags=re.MULTILINE).strip()
            beliefs = json.loads(clean)
            self._profile.beliefs.update(beliefs)
            self._profile._save()
            return beliefs
        except (json.JSONDecodeError, KeyError):
            return {}

    def run(self):
        # Full dialectic: challenge then synthesise.
        # -> dict[str, str] (updated beliefs)
        print('  [Dialectic] Challenging user model...')
        contradictions = self.challenge()
        if contradictions:
            print(f'  [Dialectic] Found {len(contradictions)} contradiction(s):')
            for c in contradictions:
                print(f'    RAW:     {c.get("raw","")[:60]}')
                print(f'    REFINED: {c.get("refined","")[:60]}')
        else:
            print('  [Dialectic] No contradictions found.')
        print('  [Dialectic] Synthesising beliefs...')
        beliefs = self.synthesise()
        return beliefs


# -- Demo: run dialectic on a profile with several signals -----------------
# Add enough signals to trigger the dialectic (MIN_SIGNALS = 4)
profile.add_signal('style', 'asked for detailed explanation of closures')
profile.add_signal('preference', 'previously said to skip explanations')

dialectic = DialecticLoop(profile)
print(f'Signals before dialectic: {dialectic._total_signals()}')
print(f'Should run: {dialectic.should_run()}')

if dialectic.should_run():
    beliefs = dialectic.run()
    print()
    print('Refined beliefs:')
    for dim, belief in beliefs.items():
        print(f'  {dim}: {belief}')

print()
print(profile.summary())

In [ ]:
class UserModeler:
    # Unified interface: wraps UserProfile + MemoryNudge + DialecticLoop.
    # One instance per user session; persists profile to disk.

    # Run dialectic every N turns (only if enough signals exist)
    DIALECTIC_EVERY_N = 5

    def __init__(self, profile_path='.user_profile.json'):
        self.profile   = UserProfile.load(path=profile_path)
        self._nudge    = MemoryNudge(self.profile)
        self._dialectic = DialecticLoop(self.profile)
        self._turn_count = 0

    def observe(self, user_message, assistant_reply):
        # Process one conversation turn: nudge + periodic dialectic.
        # user_message: str, assistant_reply: str -> bool (True if anything stored)
        self._turn_count += 1
        stored = self._nudge.nudge(user_message, assistant_reply)

        # Run dialectic periodically to refine contradictory beliefs
        if (
            self._turn_count % self.DIALECTIC_EVERY_N == 0
            and self._dialectic.should_run()
        ):
            self._dialectic.run()

        return stored

    def to_system_block(self):
        # -> str (user profile block for injection into system prompt)
        return self.profile.to_system_block()

    def summary(self):
        return (
            f'UserModeler (turn {self._turn_count}):\n'
            + self.profile.summary()
        )


# ── Update LearningAgent to incorporate UserModeler ────────────────────────
# We subclass LearningAgent to avoid duplicating all the existing code.
# This pattern shows how to extend any component in the trilogy.

class LearningAgentV2(LearningAgent):
    # Extends LearningAgent with user modeling (§9).
    # Every run() call observes the turn and injects the user profile.

    def __init__(self, profile_path='.user_profile.json', **kwargs):
        super().__init__(**kwargs)
        self.user_modeler = UserModeler(profile_path=profile_path)

    def _build_system_with_user(self):
        # Merge the base system prompt with the current user profile block.
        # -> str (system prompt + user profile section)
        user_block = self.user_modeler.to_system_block()
        if not user_block:
            return self.current_system
        return self.current_system + '\n\n' + user_block

    def learn_and_run(
        self,
        task,
        max_reflexion_attempts=2,
        n_self_reward_candidates=2,
        run_skill_audit=False,
    ):
        # Override: inject user profile into system prompt before each run,
        # then observe the turn afterwards.
        # All other behaviour inherited from LearningAgent.

        # Temporarily patch the current system prompt with user profile
        original_system = self.current_system
        self.current_system = self._build_system_with_user()
        self._reflexion._base_system = self.current_system

        reply = super().learn_and_run(
            task,
            max_reflexion_attempts=max_reflexion_attempts,
            n_self_reward_candidates=n_self_reward_candidates,
            run_skill_audit=run_skill_audit,
        )

        # Restore base system (user block is rebuilt fresh each turn)
        self.current_system = original_system
        self._reflexion._base_system = original_system

        # Observe the completed turn to update user model
        self.user_modeler.observe(task, reply)

        return reply

    def status(self):
        return super().status() + '\n' + self.user_modeler.summary()


# ── Full Capstone Demo: LearningAgentV2 -------------------------------------
print('Initialising LearningAgentV2 (with user modeling)...')
agent_v2 = LearningAgentV2(
    profile_path='.demo_v2_profile.json',
    base_system='You are a helpful, precise assistant.',
    prompt_opt_every_n=20,  # high to avoid triggering in demo
)

# Simulate a short user session with personality cues
session_tasks = [
    'just the command to list files including hidden ones',
    'what does torch.nn.Module.parameters() return exactly',
    'i am working on a transformer for protein folding, how should i set d_model',
]

for task in session_tasks:
    reply = agent_v2.learn_and_run(
        task,
        max_reflexion_attempts=1,
        n_self_reward_candidates=1,   # speed
    )
    print(f'  Reply: {reply[:80]}')

print()
print(agent_v2.status())
print()
print('User profile system block:')
print(agent_v2.user_modeler.to_system_block())

# Summary

## The Self-Improvement Taxonomy

| Mechanism | What improves | Persistence | Requires external oracle? |
|-----------|--------------|-------------|--------------------------|
| **Reflexion (§1)** | Episodic strategy | Session | No |
| **Prompt Optimisation (§2)** | System prompt | Yes | Benchmark examples |
| **Skill Verification (§3)** | Skill quality | Yes | Test cases in skill |
| **Meta-Agent / ADAS (§4)** | Agent architecture | Yes | Benchmark tasks |
| **Research Loop (§5)** | Domain knowledge | Session | No (self-designed) |
| **Darwin Godel Machine (§6)** | Source code | Yes (recursive) | Benchmark oracle |
| **Self-Rewarding (§7)** | Per-response quality | No | No -- fully self-contained |

## Where Each Mechanism Lives in the Learning Hierarchy

```
Level 3: Architecture  --- Darwin Godel Machine, Meta-Agent/ADAS
                              (agent redesigns itself)
         |
Level 2: Knowledge     --- Research Loop, Skill Library, FTS5 Search
                              (agent accumulates knowledge)
         |
Level 1: Behaviour     --- Reflexion, Prompt Optimisation, Self-Reward
                              (agent adjusts outputs without structural change)
```

## Further Reading

- Shinn et al., *Reflexion*, NeurIPS 2023
- Fernando et al., *PromptBreeder*, ICML 2024
- Wang et al., *Voyager*, NeurIPS 2023
- Hu et al., *ADAS: Automated Design of Agentic Systems*, 2025
- Zhang et al., *Darwin Godel Machine*, arXiv 2505.22954, 2025
- Yuan et al., *Self-Rewarding Language Models*, 2025
- Khattab et al., *DSPy*, arXiv 2310.03714, 2023